# SarcasTone - Notebook 01: Phase 1 (Text)

Reproduces the Phase 1 text ladder on the **locked** MUStARD++ splits, then the champion.

| Stage | Model | Config / command |
|---|---|---|
| A1 | TF-IDF (1,2)-grams + LogReg | `text_baseline --mode tfidf` |
| A2 | frozen `bert-base-uncased` CLS + LogReg | `text_baseline --mode bert_cls` |
| B  | fine-tuned BERT | `bert_finetune --config configs/text_bert.yaml` |
| C  | fine-tuned RoBERTa (5 ep / 3e-5, no context) | `bert_finetune --config configs/tune_roberta_long.yaml` |
| D  | News-Headlines booster -> MUStARD (champion) | `boost_headlines` then `boost_mustard` (`--tag roberta_boosted`) |

**Reference champion:** `roberta_boosted` test macro-F1 = **0.6866** (acc 0.6923).

Honest note: at n=104 a single-split F1 has ~±0.036 spread (5-fold CV). Differences of
0.02-0.03 are noise. We never tune against the test set.

In [ ]:
# ---- standalone bootstrap (safe to re-run) ----
import os, sys
PROJECT = '/content/SarcasTone'
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
sys.path.insert(0, os.path.abspath('src'))
import sarcastone
print('cwd =', os.getcwd())

# heavy stages are gated so the notebook can be re-run cheaply
RUN_BASELINES = True
RUN_BERT      = True
RUN_ROBERTA   = True
RUN_BOOSTER   = True   # needs the LFS checkpoint (see 00_setup step 3)

## 0. Snapshot the committed reference metrics

The repo ships the authoritative reports. Capture them **before** re-running so we can
diff fresh results against them (re-running overwrites the same JSON paths).

In [ ]:
import json
from pathlib import Path
from sarcastone.utils import REPORTS_DIR

TAGS = ['lrtfidf', 'lrbertcls', 'bert', 'roberta', 'roberta_nh', 'roberta_boosted']
before = {}
for t in TAGS:
    p = Path(REPORTS_DIR) / f'phase1_{t}_test_metrics.json'
    before[t] = json.loads(p.read_text()) if p.exists() else None
for t, d in before.items():
    print(f'{t:16s}', (f"F1={d['f1_macro']:.4f} acc={d['accuracy']:.4f}" if d else 'MISSING'))

## A. Baselines (no fine-tuning)

Cheap sanity floor. Writes `phase1_lrtfidf_*` and `phase1_lrbertcls_*` artifacts + confusion plots.

In [ ]:
if RUN_BASELINES:
    !python -m sarcastone.models.text_baseline --mode both

## B. Fine-tuned BERT (`bert-base-uncased`, 3 ep / 2e-5)

Dumps `[CLS]` embeddings for Phase 3 (`checkpoints/text_bert`, `embeddings/text_*`).

In [ ]:
if RUN_BERT:
    !python -m sarcastone.models.bert_finetune --config configs/text_bert.yaml

## C. Fine-tuned RoBERTa (no context, 5 ep / 3e-5)

Winning base recipe (context hurt here). Produces `checkpoints/text_roberta`.

In [ ]:
if RUN_ROBERTA:
    !python -m sarcastone.models.bert_finetune --config configs/tune_roberta_long.yaml

## D. Booster -> champion

Stage 1: intermediate fine-tune on ~28.6k News-Headlines (`checkpoints/text_roberta_nh`).
Stage 2: fine-tune that checkpoint on MUStARD++ = **champion** (`--tag roberta_boosted`).

Stage 1 is skipped automatically here; its output ships via LFS. To retrain it, see
`00_setup` step 6 and run `configs/boost_headlines.yaml` first.

In [ ]:
if RUN_BOOSTER:
    # Stage 1 (optional, needs Kaggle corpus):
    # !python -m sarcastone.models.bert_finetune --config configs/boost_headlines.yaml
    # Stage 2 -> champion:
    !python -m sarcastone.models.bert_finetune --config configs/boost_mustard.yaml --tag roberta_boosted

## Verify: fresh vs committed

Identical numbers confirm the environment reproduces the committed results exactly.

In [ ]:
rows = []
for t in TAGS:
    p = Path(REPORTS_DIR) / f'phase1_{t}_test_metrics.json'
    after = json.loads(p.read_text()) if p.exists() else None
    b = before.get(t); a = after
    rows.append((t,
                 b['f1_macro'] if b else None,
                 a['f1_macro'] if a else None,
                 round((a['f1_macro'] - b['f1_macro']), 4) if b and a else None))
import pandas as pd
print(pd.DataFrame(rows, columns=['tag', 'committed_F1', 'fresh_F1', 'delta']).to_string(index=False))

## Significance: champion vs BERT on the locked test

Exact McNemar (paired accuracy) + paired bootstrap CI on macro-F1. Both models predict the
**same** locked test set, so the tests are valid. At n=104 expect wide CIs.

In [ ]:
import numpy as np, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sarcastone.utils import SPLITS_DIR
from sarcastone.evaluation.metrics import compute_metrics
from sarcastone.evaluation.significance import mcnemar_exact, paired_bootstrap_f1

test = pd.read_csv(Path(SPLITS_DIR) / 'test.csv')
y = test.label.values

@torch.no_grad()
def probs_for(ckpt, batch=32, max_len=128):
    tok   = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).eval()
    texts = test.text.tolist()
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], padding=True, truncation=True,
                  max_length=max_len, return_tensors='pt')
        out.extend(torch.softmax(model(**enc).logits, -1)[:, 1].tolist())
    return np.array(out)

ckpts = {'champion (roberta_boosted)': 'checkpoints/text_roberta_boosted',
         'bert': 'checkpoints/text_bert'}
probs = {}
for name, ck in ckpts.items():
    if os.path.isdir(ck):
        probs[name] = probs_for(ck)
        print(f"{name:28s} test F1={compute_metrics(y, (probs[name]>=0.5).astype(int))['f1_macro']:.4f}")
    else:
        print(f'{name:28s} checkpoint missing: {ck}')

In [ ]:
if len(probs) == 2:
    (na, pa), (nb, pb) = probs.items()
    pa_bin = (pa >= 0.5).astype(int); pb_bin = (pb >= 0.5).astype(int)
    print('McNemar (exact):', mcnemar_exact(y, pa_bin, pb_bin))
    print('Bootstrap dF1  :', paired_bootstrap_f1(y, pa, pb))

## Honest reading

- Single-split differences below ~0.03 macro-F1 are **not** reliable at n=104.
- The champion's 0.6866 partly reflects a favourable split; the 5-fold mean is ~0.626.
- Data expansion (adding MUStARD++ full as training-only) was tested and did **not** help
  the locked test (0.6866 -> 0.6664); the extra clips are a different distribution.
- The gate (test F1 >= 0.70) is therefore not reached by text alone; it is a stretch target,
  and the scientifically valid claim is the multimodal gain (Phase 3), not the point estimate.